# 🪙 Crypto Price Predictor — LSTM + 44 Features Engineering

**Flujo completo:**
1. Actualiza datos raw hasta hoy (OHLCV + Futures: funding rate, OI, long/short)
2. 📊 **Análisis de Series Temporales**
3. Construye 44 features curadas por grupo semántico
4. Entrena LSTM puro con atención
5. Backtesting con comisiones reales
6. Inferencia con intervalos de confianza

> **GPU T4** activado en Settings → Accelerator

In [ ]:
!pip install torch pandas numpy scikit-learn matplotlib statsmodels --quiet

import os, math, time, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.tsa.seasonal import seasonal_decompose
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
import requests
from datetime import datetime, timezone

warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

RAW_DIR    = "/kaggle/input/datasets/vicentelorenzomarn/dataraw"
WORK_DIR   = "/kaggle/working"
MODELS_DIR = "/kaggle/input/datasets/vicentelorenzomarn/crypto-models"
os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs("plots", exist_ok=True)

MAPE_HISTORICO = {
    "BTC":  0.0055, "ETH":  0.0091, "SOL":  0.0101,
    "AVAX": 0.0106,
}
UMBRALES_BT = {
    "BTC": 0.003, "ETH": 0.003, "SOL": 0.003,
    "AVAX": 0.003,
}


# ⚙️ Configuración — 44 Features Curadas

In [ ]:
# ──────────────────────────────────────────────────────────────
# FEATURE_COLS — 44 features semánticamente agrupadas
# Filosofía: calidad > cantidad. Cada feature aporta señal única.
# ──────────────────────────────────────────────────────────────
FEATURE_COLS = [
    # ── Posición de precio (6) ────────────────────────────────
    # Ratios relativas a EMAs: estables en cualquier régimen de precio
    "close_vs_ema7",    # close/ema7 - 1   → sesgo de muy corto plazo
    "close_vs_ema14",   # close/ema14 - 1  → sesgo de corto plazo
    "close_vs_ema50",   # close/ema50 - 1  → sesgo de medio plazo
    "return_1",         # retorno 1h
    "return_4",         # retorno 4h
    "return_24",        # retorno 24h

    # ── Momentum: lags de autocorrelación (4) ─────────────────
    "return_lag_1",     # retorno de hace 1h  → autocorrelación lag-1
    "return_lag_2",
    "return_lag_3",
    "return_lag_4",

    # ── Indicadores técnicos (8) ──────────────────────────────
    "rsi_14",           # RSI 14 periodos
    "rsi_6",            # RSI 6 periodos (más reactivo)
    "macd",             # MACD línea rápida
    "macd_signal",      # MACD señal
    "macd_hist",        # MACD histograma = macd - macd_signal
    "bb_position",      # posición dentro de Bandas de Bollinger
    "bb_width",         # ancho de Bollinger (volatilidad implícita)
    "atr_14_norm",      # ATR 14 / close  → volatilidad normalizada

    # ── Volumen (2) ───────────────────────────────────────────
    "volume_ratio",     # volumen vs media 24h
    "volume_ratio_4h",  # volumen vs media 4h (más reactivo)

    # ── Estructura de mercado (6) ─────────────────────────────
    "range_pos_24h",    # posición en rango max-min de 24h
    "range_pos_7d",     # posición en rango max-min de 7 días
    "dist_to_high_24h", # distancia relativa al máximo 24h
    "dist_to_low_24h",  # distancia relativa al mínimo 24h
    "volatility_24h",   # std de retornos en 24h
    "vol_ratio",        # volatility_24h / volatility_7d (régimen de vol)

    # ── Tiempo cíclico (6) ────────────────────────────────────
    "hour_sin", "hour_cos",   # hora del día
    "dow_sin",  "dow_cos",    # día de la semana
    "month_sin","month_cos",  # mes del año

    # ── Multi-timeframe (4) ───────────────────────────────────
    "return_4h",    # retorno de la vela 4h más reciente (sin lookahead)
    "rsi_4h",       # RSI 14 en 4h
    "return_1d",    # retorno de la vela diaria más reciente
    "rsi_1d",       # RSI 14 en 1d

    # ── Datos de derivados de Binance Futures (4) ─────────────
    "funding_rate",         # tasa de financiación (señal de sentimiento)
    "funding_rate_24h_avg", # media 24h de funding rate
    "oi_change_4h",         # cambio en Open Interest en 4h
    "long_short_ratio",     # ratio long/short cuentas globales

    # ── Sentimiento externo (1) ───────────────────────────────
    "fear_greed",           # Fear & Greed Index (0-100)

    # ── Cross-asset BTC (3) — solo ETH/SOL/AVAX ──────────────
    # Para BTC se rellena con 0 (él mismo)
    "btc_return_1",   # retorno 1h de BTC
    "btc_return_4h",  # retorno 4h de BTC
    "btc_rsi_14",     # RSI 14 de BTC
]

print(f"Total features → {len(FEATURE_COLS)}")
assert len(FEATURE_COLS) == 44, f"Se esperaban 44 features, hay {len(FEATURE_COLS)}"

BASE_CONFIG = {
    "data_dir":        WORK_DIR,
    "raw_dir":         RAW_DIR,
    "coins":           ["BTC", "ETH", "SOL", "AVAX"],
    "granularity":     "hourly",
    "horizons":        [1, 2, 3, 4],
    "horizon_weights": [1.0, 0.85, 0.70, 0.55],
    "train_ratio":     0.70,
    "val_ratio":       0.15,
    "feature_cols":    FEATURE_COLS,
    "epochs":          150,
    "patience":        20,
    "wf_folds":        5,
}

# Configs LSTM por moneda (arquitecturas optimizadas para 44 features)
BEST_CONFIGS = {
    "BTC": {**BASE_CONFIG, **{
        "lstm_hidden":    256, "lstm_layers":    2,
        "lstm_dropout":   0.145, "head_dropout":   0.275,
        "head_hidden":    128, "seq_len":        24,
        "batch_size":     64, "learning_rate":  0.00174,
        "weight_decay":   0.0327,
    }},
    "ETH": {**BASE_CONFIG, **{
        "lstm_hidden":    128, "lstm_layers":    3,
        "lstm_dropout":   0.134, "head_dropout":   0.284,
        "head_hidden":    128, "seq_len":        24,
        "batch_size":     32, "learning_rate":  0.00172,
        "weight_decay":   0.00139,
    }},
    "SOL": {**BASE_CONFIG, **{
        "lstm_hidden":    256, "lstm_layers":    2,
        "lstm_dropout":   0.29,  "head_dropout":   0.24,
        "head_hidden":    128, "seq_len":        24,
        "batch_size":     128, "learning_rate":  0.000190,
        "weight_decay":   0.000436,
    }},
    "AVAX": {**BASE_CONFIG, **{
        "lstm_hidden":    128, "lstm_layers":    2,
        "lstm_dropout":   0.32,  "head_dropout":   0.13,
        "head_hidden":    64,  "seq_len":        48,
        "batch_size":     128, "learning_rate":  0.000286,
        "weight_decay":   0.00324,
    }},
}

print("✅ Configuración cargada.")


# 📥 Descarga y Actualización de Datos OHLCV

In [ ]:
COINS_MAP = {
    "BTC": "BTCUSDT", "ETH": "ETHUSDT", "SOL": "SOLUSDT",
    "AVAX": "AVAXUSDT",
}
START_DATES = {
    "BTC": "2020-01-01", "ETH": "2020-01-01", "SOL": "2020-09-01",
    "AVAX": "2020-10-01",
}
BINANCE_SPOT_URLS = [
    "https://api.binance.us/api/v3/klines",
    "https://api.binance.com/api/v3/klines",
]
BINANCE_FUTURES_BASE = "https://fapi.binance.com"

def _date_to_ms(date_str):
    return int(datetime.strptime(date_str, "%Y-%m-%d")
               .replace(tzinfo=timezone.utc).timestamp() * 1000)

def binance_get(params, timeout=15, urls=None):
    urls = urls or BINANCE_SPOT_URLS
    for url in urls:
        try:
            r = requests.get(url, params=params, timeout=timeout)
            r.raise_for_status()
            return r.json()
        except Exception:
            continue
    return None

def download_historical(symbol, interval, start_date):
    start_ms = _date_to_ms(start_date)
    end_ms   = int(datetime.now(timezone.utc).timestamp() * 1000)
    rows     = []
    while start_ms < end_ms:
        batch = binance_get({"symbol": symbol, "interval": interval,
                              "startTime": start_ms, "limit": 1000})
        if not batch: break
        for c in batch:
            rows.append({"timestamp": pd.to_datetime(c[0], unit="ms", utc=True),
                          "open": float(c[1]), "high": float(c[2]),
                          "low":  float(c[3]), "close": float(c[4]),
                          "volume": float(c[5])})
        start_ms = batch[-1][0] + 1
        time.sleep(0.15)
    if not rows: return None
    return (pd.DataFrame(rows).drop_duplicates("timestamp")
              .sort_values("timestamp").reset_index(drop=True))

def download_recent(symbol, interval, limit=500):
    batch = binance_get({"symbol": symbol, "interval": interval, "limit": limit})
    if not batch: return None
    rows = [{"timestamp": pd.to_datetime(c[0], unit="ms", utc=True),
             "open": float(c[1]), "high": float(c[2]),
             "low":  float(c[3]), "close": float(c[4]),
             "volume": float(c[5])} for c in batch]
    return (pd.DataFrame(rows).drop_duplicates("timestamp")
              .sort_values("timestamp").reset_index(drop=True))

def update_raw_csv(path, symbol, interval, start_date, coin):
    if os.path.exists(path):
        df_existing = pd.read_csv(path, parse_dates=["timestamp"])
        if df_existing["timestamp"].dt.tz is None:
            df_existing["timestamp"] = df_existing["timestamp"].dt.tz_localize("UTC")
        last_ts = df_existing["timestamp"].max()
        df_new = download_recent(symbol, interval, limit=1000)
        if df_new is not None:
            if df_new["timestamp"].dt.tz is None:
                df_new["timestamp"] = df_new["timestamp"].dt.tz_localize("UTC")
            df_new = df_new[df_new["timestamp"] > last_ts]
        if df_new is None or len(df_new) == 0:
            print(f"  {coin} {interval}: ya actualizado ({last_ts.date()}) ✅")
            return
        df_new["coin"] = coin
        df_updated = (pd.concat([df_existing, df_new], ignore_index=True)
                        .drop_duplicates("timestamp")
                        .sort_values("timestamp").reset_index(drop=True))
        df_updated.to_csv(path, index=False)
        print(f"  {coin} {interval}: +{len(df_new)} velas → hasta {df_updated['timestamp'].max().date()} ✅")
    else:
        src_candidates = [
            f"{RAW_DIR}/{coin}_{interval}_raw.csv",
            f"{RAW_DIR}/{coin}_hourly_raw.csv" if interval == "1h" else None,
        ]
        copied = False
        for src in src_candidates:
            if src and os.path.exists(src):
                df = pd.read_csv(src, parse_dates=["timestamp"])
                if df["timestamp"].dt.tz is None:
                    df["timestamp"] = df["timestamp"].dt.tz_localize("UTC")
                df["coin"] = coin
                df.to_csv(path, index=False)
                print(f"  {coin} {interval}: copiado desde dataset ✅")
                copied = True
                update_raw_csv(path, symbol, interval, start_date, coin)
                break
        if copied: return
        df = download_historical(symbol, interval, start_date)
        if df is not None:
            df["coin"] = coin
            df.to_csv(path, index=False)
            print(f"{len(df):,} velas → hasta {df['timestamp'].max().date()} ✅")
        else:
            print("✗ no disponible")

def ensure_and_update_all(work_dir):
    print("\n── Actualizando datos raw OHLCV ──────────────────────────────")
    for coin, symbol in COINS_MAP.items():
        for interval in ["1h", "4h", "1d"]:
            path = f"{work_dir}/{coin}_{interval}_raw.csv"
            update_raw_csv(path, symbol, interval, START_DATES[coin], coin)
    print("✅ Datos OHLCV actualizados.\n")

# ensure_and_update_all(WORK_DIR)


# 📡 Datos de Derivados: Funding Rate, Open Interest, Long/Short Ratio

In [ ]:
# ──────────────────────────────────────────────────────────────
# Binance Futures endpoints para derivados
# Estos datos capturan el sentimiento del mercado de futuros,
# que tiene alto poder predictivo para movimientos de precio.
# ──────────────────────────────────────────────────────────────

def download_funding_rate(symbol, limit=1000):
    """
    Descarga historial de funding rates de Binance Futures.
    Funding rate > 0 → longs pagan a shorts → mercado sobrecalentado (bajista)
    Funding rate < 0 → shorts pagan a longs → pánico vendedor (alcista)
    """
    url = f"{BINANCE_FUTURES_BASE}/fapi/v1/fundingRate"
    rows = []
    end_time = int(datetime.now(timezone.utc).timestamp() * 1000)
    while True:
        try:
            resp = requests.get(url, params={
                "symbol": symbol, "limit": limit, "endTime": end_time
            }, timeout=15)
            resp.raise_for_status()
            batch = resp.json()
            if not batch: break
            for r in batch:
                rows.append({
                    "timestamp": pd.to_datetime(r["fundingTime"], unit="ms", utc=True),
                    "funding_rate": float(r["fundingRate"]),
                })
            end_time = batch[0]["fundingTime"] - 1
            time.sleep(0.1)
            if len(batch) < limit: break
        except Exception as e:
            print(f"  ⚠ Funding rate {symbol}: {e}")
            break
    if not rows:
        return pd.DataFrame(columns=["timestamp", "funding_rate"])
    df = (pd.DataFrame(rows)
            .drop_duplicates("timestamp")
            .sort_values("timestamp")
            .reset_index(drop=True))
    print(f"  Funding rate {symbol}: {len(df)} registros ✅")
    return df

def download_open_interest_hist(symbol, period="1h", limit=500):
    """
    Descarga historial de Open Interest (interés abierto).
    OI creciente con precio subiendo → momentum alcista genuino
    OI cayendo con precio subiendo → rally sin convicción
    """
    url = f"{BINANCE_FUTURES_BASE}/futures/data/openInterestHist"
    rows = []
    end_time = int(datetime.now(timezone.utc).timestamp() * 1000)
    while True:
        try:
            resp = requests.get(url, params={
                "symbol": symbol, "period": period,
                "limit": limit, "endTime": end_time,
            }, timeout=15)
            resp.raise_for_status()
            batch = resp.json()
            if not batch or not isinstance(batch, list): break
            for r in batch:
                rows.append({
                    "timestamp": pd.to_datetime(r["timestamp"], unit="ms", utc=True),
                    "open_interest": float(r["sumOpenInterest"]),
                    "open_interest_usd": float(r["sumOpenInterestValue"]),
                })
            end_time = batch[0]["timestamp"] - 1
            time.sleep(0.1)
            if len(batch) < limit: break
        except Exception as e:
            print(f"  ⚠ OI {symbol}: {e}")
            break
    if not rows:
        return pd.DataFrame(columns=["timestamp", "open_interest", "open_interest_usd"])
    df = (pd.DataFrame(rows)
            .drop_duplicates("timestamp")
            .sort_values("timestamp")
            .reset_index(drop=True))
    print(f"  Open Interest {symbol}: {len(df)} registros ✅")
    return df

def download_long_short_ratio(symbol, period="1h", limit=500):
    """
    Descarga ratio long/short de cuentas globales.
    > 1.0 → mayoría de cuentas están long
    < 1.0 → mayoría de cuentas están short
    Extremos en cualquier dirección son señal contraria.
    """
    url = f"{BINANCE_FUTURES_BASE}/futures/data/globalLongShortAccountRatio"
    rows = []
    end_time = int(datetime.now(timezone.utc).timestamp() * 1000)
    while True:
        try:
            resp = requests.get(url, params={
                "symbol": symbol, "period": period,
                "limit": limit, "endTime": end_time,
            }, timeout=15)
            resp.raise_for_status()
            batch = resp.json()
            if not batch or not isinstance(batch, list): break
            for r in batch:
                rows.append({
                    "timestamp": pd.to_datetime(int(r["timestamp"]), unit="ms", utc=True),
                    "long_short_ratio": float(r["longShortRatio"]),
                })
            end_time = int(batch[0]["timestamp"]) - 1
            time.sleep(0.1)
            if len(batch) < limit: break
        except Exception as e:
            print(f"  ⚠ Long/Short {symbol}: {e}")
            break
    if not rows:
        return pd.DataFrame(columns=["timestamp", "long_short_ratio"])
    df = (pd.DataFrame(rows)
            .drop_duplicates("timestamp")
            .sort_values("timestamp")
            .reset_index(drop=True))
    print(f"  Long/Short ratio {symbol}: {len(df)} registros ✅")
    return df

def build_derivatives_df(symbol, coin, work_dir, force=False):
    """
    Construye un DataFrame horario con funding_rate, oi_change_4h y long_short_ratio.
    Guarda en caché en disco para evitar re-descargas.
    """
    cache_path = f"{work_dir}/{coin}_derivatives.csv"
    if os.path.exists(cache_path) and not force:
        df = pd.read_csv(cache_path, parse_dates=["timestamp"])
        if df["timestamp"].dt.tz is None:
            df["timestamp"] = df["timestamp"].dt.tz_localize("UTC")
        # Actualizar si hay datos nuevos
        last = df["timestamp"].max()
        now  = pd.Timestamp.now(tz="UTC")
        if (now - last).total_seconds() / 3600 < 2:
            print(f"  {coin} derivatives: ya actualizados ✅")
            return df
    
    print(f"  Descargando derivados {coin}...")
    
    # 1. Funding rate (cada 8h en Binance)
    df_fr = download_funding_rate(symbol)
    
    # 2. Open Interest horario
    df_oi = download_open_interest_hist(symbol)
    
    # 3. Long/Short ratio horario
    df_ls = download_long_short_ratio(symbol)
    
    # Obtener el rango de fechas de OHLCV para alinear
    ohlcv_path = f"{work_dir}/{coin}_1h_raw.csv"
    if not os.path.exists(ohlcv_path):
        ohlcv_path = f"{RAW_DIR}/{coin}_1h_raw.csv"
    df_ohlcv = pd.read_csv(ohlcv_path, usecols=["timestamp"], parse_dates=["timestamp"])
    if df_ohlcv["timestamp"].dt.tz is None:
        df_ohlcv["timestamp"] = df_ohlcv["timestamp"].dt.tz_localize("UTC")
    
    # Crear DataFrame hourly alineado con el OHLCV
    hourly_ts = df_ohlcv["timestamp"].sort_values().reset_index(drop=True)
    df_base = pd.DataFrame({"timestamp": hourly_ts})
    
    # Merge funding rate (se actualiza cada 8h, rellenar hacia adelante)
    if len(df_fr) > 0:
        df_base = pd.merge_asof(
            df_base.sort_values("timestamp"),
            df_fr.sort_values("timestamp"),
            on="timestamp", direction="backward"
        )
        df_base["funding_rate"] = df_base["funding_rate"].ffill().fillna(0.0)
    else:
        df_base["funding_rate"] = 0.0
    
    # Media 24h de funding rate
    # Primero interpolamos el funding rate a frecuencia horaria para la media móvil
    df_base["funding_rate_24h_avg"] = df_base["funding_rate"].rolling(24, min_periods=1).mean()
    
    # Merge Open Interest
    if len(df_oi) > 0:
        df_base = pd.merge_asof(
            df_base.sort_values("timestamp"),
            df_oi[["timestamp", "open_interest"]].sort_values("timestamp"),
            on="timestamp", direction="backward"
        )
        df_base["open_interest"] = df_base["open_interest"].ffill().fillna(method="bfill")
        # OI change en 4h: variación porcentual del OI en las últimas 4 velas
        df_base["oi_change_4h"] = df_base["open_interest"].pct_change(4).clip(-0.5, 0.5).fillna(0.0)
    else:
        df_base["open_interest"] = 0.0
        df_base["oi_change_4h"]  = 0.0
    
    # Merge Long/Short ratio
    if len(df_ls) > 0:
        df_base = pd.merge_asof(
            df_base.sort_values("timestamp"),
            df_ls.sort_values("timestamp"),
            on="timestamp", direction="backward"
        )
        df_base["long_short_ratio"] = df_base["long_short_ratio"].ffill().fillna(1.0)
    else:
        df_base["long_short_ratio"] = 1.0
    
    # Guardar caché
    cols = ["timestamp", "funding_rate", "funding_rate_24h_avg", "oi_change_4h", "long_short_ratio"]
    df_out = df_base[cols].sort_values("timestamp").reset_index(drop=True)
    df_out.to_csv(cache_path, index=False)
    print(f"  {coin} derivatives: {len(df_out)} filas guardadas ✅")
    return df_out

def download_fear_greed(limit=2000, verbose=False):
    try:
        resp = requests.get(
            f"https://api.alternative.me/fng/?limit={limit}&format=json", timeout=15
        )
        resp.raise_for_status()
        data = resp.json()["data"]
        df = pd.DataFrame([{
            "date": pd.to_datetime(int(d["timestamp"]), unit="s", utc=True).normalize(),
            "fear_greed": int(d["value"])
        } for d in data]).sort_values("date").reset_index(drop=True)
        if verbose: print(f"  Fear & Greed: {len(df)} días (hasta {df['date'].max().date()}) ✅")
        return df
    except Exception as e:
        if verbose: print(f"  ⚠ Fear & Greed no disponible ({e}) — usando 50")
        dates = pd.date_range("2020-01-01", periods=2500, freq="D", tz="UTC")
        return pd.DataFrame({"date": dates, "fear_greed": 50})

FG_DF = download_fear_greed(verbose=True)
print("✅ Datos de sentimiento listos.")


# 📊 Análisis de Series Temporales

In [ ]:
def load_raw_for_analysis(coin, raw_dir):
    path = f"{raw_dir}/{coin}_1h_raw.csv"
    df = pd.read_csv(path, parse_dates=["timestamp"]).sort_values("timestamp")
    df["return"] = df["close"].pct_change()
    df = df.dropna().reset_index(drop=True)
    return df

def test_stationarity(series, name="Serie"):
    result = adfuller(series.dropna(), autolag="AIC")
    print(f"  {name}")
    print(f"    ADF Statistic : {result[0]:.4f}")
    print(f"    p-value       : {result[1]:.6f}  {'✅ ESTACIONARIA' if result[1] < 0.05 else '❌ NO ESTACIONARIA'}")
    print()
    return result[1] < 0.05

def run_stationarity_analysis(coins, raw_dir):
    print("\n" + "="*65)
    print("  TEST DE ESTACIONARIEDAD (ADF) — Precio vs Retorno")
    print("="*65 + "\n")
    for coin in coins:
        try:
            df = load_raw_for_analysis(coin, raw_dir)
            print(f"─── {coin} ─────────────────────────────────────")
            test_stationarity(df["close"].tail(5000), f"Precio cierre (últimas 5000h)")
            test_stationarity(df["return"].tail(5000), f"Retorno (pct_change)")
        except FileNotFoundError:
            print(f"  {coin}: archivo no encontrado, omitiendo")

run_stationarity_analysis(BASE_CONFIG["coins"], RAW_DIR)


In [ ]:
def plot_acf_pacf(coin, raw_dir, lags=48, col="return"):
    try:
        df = load_raw_for_analysis(coin, raw_dir)
        series = df[col].tail(5000).dropna().values
        acf_vals  = acf(series,  nlags=lags, fft=True)
        pacf_vals = pacf(series, nlags=lags, method="ywm")
        ci = 1.96 / np.sqrt(len(series))
        lags_x = np.arange(1, lags + 1)
        fig, axes = plt.subplots(1, 2, figsize=(14, 4))
        fig.suptitle(f"{coin} — ACF y PACF del Retorno Horario", fontsize=12, fontweight="bold")
        for ax, vals, title, color in [
            (axes[0], acf_vals[1:],  "ACF — ¿Hasta qué lag hay memoria?",  "steelblue"),
            (axes[1], pacf_vals[1:], "PACF — Efecto directo de cada lag",   "coral"),
        ]:
            ax.bar(lags_x, vals, color=color, alpha=0.7, width=0.6)
            ax.axhline( ci, color="red", linestyle="--", linewidth=0.9, label=f"IC 95% (±{ci:.3f})")
            ax.axhline(-ci, color="red", linestyle="--", linewidth=0.9)
            ax.axhline( 0,  color="black", linewidth=0.5)
            ax.set_xlim(0, lags + 1)
            max_visible = max(np.abs(vals).max() * 1.3, ci * 3)
            ax.set_ylim(-max_visible, max_visible)
            ax.set_title(title); ax.set_xlabel("Lag (horas)"); ax.legend(fontsize=8); ax.grid(alpha=0.3)
            top3_idx = np.argsort(np.abs(vals))[-3:][::-1]
            for idx in top3_idx:
                if abs(vals[idx]) > ci:
                    ax.annotate(f"lag {lags_x[idx]}", xy=(lags_x[idx], vals[idx]),
                                xytext=(0, 6 * np.sign(vals[idx])), textcoords="offset points",
                                ha="center", fontsize=7, color="darkred")
        plt.tight_layout()
        plt.savefig(f"plots/{coin}_acf_pacf.png", dpi=120, bbox_inches="tight")
        plt.show()
        print(f"  {coin}: ACF/PACF guardado ✅")
    except FileNotFoundError:
        print(f"  {coin}: archivo no encontrado")

print("\nGenerando gráficos ACF/PACF")
for coin in BASE_CONFIG["coins"]:
    plot_acf_pacf(coin, RAW_DIR)


# ⚙️ Feature Engineering — 44 Features Curadas

In [ ]:
# ──────────────────────────────────────────────────────────────
# calc_indicators: calcula todos los indicadores técnicos base
# ──────────────────────────────────────────────────────────────
def calc_indicators(df):
    df = df.copy()
    
    # ── EMAs (3 periodos) ─────────────────────────────────────
    df["ema_7"]  = df["close"].ewm(span=7,  adjust=False).mean()
    df["ema_14"] = df["close"].ewm(span=14, adjust=False).mean()
    df["ema_50"] = df["close"].ewm(span=50, adjust=False).mean()  # [NEW] EMA 50
    
    # ── RSI 14 ────────────────────────────────────────────────
    delta = df["close"].diff()
    ag14  = delta.clip(lower=0).ewm(alpha=1/14, min_periods=14, adjust=False).mean()
    al14  = (-delta).clip(lower=0).ewm(alpha=1/14, min_periods=14, adjust=False).mean()
    df["rsi_14"] = 100 - 100 / (1 + ag14 / (al14 + 1e-9))
    
    # ── RSI 6 (más reactivo) ──────────────────────────────────
    # [NEW] RSI de corto plazo — captura sobrecompra/sobreventa rápida
    ag6  = delta.clip(lower=0).ewm(alpha=1/6,  min_periods=6,  adjust=False).mean()
    al6  = (-delta).clip(lower=0).ewm(alpha=1/6,  min_periods=6,  adjust=False).mean()
    df["rsi_6"] = 100 - 100 / (1 + ag6 / (al6 + 1e-9))
    
    # ── MACD (12-26-9) ────────────────────────────────────────
    ema12 = df["close"].ewm(span=12, adjust=False).mean()
    ema26 = df["close"].ewm(span=26, adjust=False).mean()
    df["macd"]        = ema12 - ema26
    df["macd_signal"] = df["macd"].ewm(span=9, adjust=False).mean()
    df["macd_hist"]   = df["macd"] - df["macd_signal"]  # [NEW] histograma MACD
    
    # ── Bandas de Bollinger (20 periodos, 2σ) ─────────────────
    sma20 = df["close"].rolling(20).mean()
    std20 = df["close"].rolling(20).std()
    bb_u  = sma20 + 2 * std20
    bb_l  = sma20 - 2 * std20
    df["bb_width"]    = (bb_u - bb_l) / (sma20 + 1e-9)
    df["bb_position"] = (df["close"] - bb_l) / (bb_u - bb_l + 1e-9)
    
    # ── ATR 14 ────────────────────────────────────────────────
    hl = df["high"] - df["low"]
    hc = (df["high"] - df["close"].shift(1)).abs()
    lc = (df["low"]  - df["close"].shift(1)).abs()
    df["atr_14"] = (pd.concat([hl, hc, lc], axis=1).max(axis=1)
                    .ewm(alpha=1/14, min_periods=14, adjust=False).mean())
    
    return df


def add_features_1h(df):
    """
    Construye todas las features sobre el timeframe 1h.
    Grupos: posición de precio, momentum, indicadores técnicos,
            volumen, estructura de mercado, tiempo.
    """
    df = df.copy()
    if df["timestamp"].dt.tz is None:
        df["timestamp"] = df["timestamp"].dt.tz_localize("UTC")
    
    df = calc_indicators(df)
    
    # ── Grupo: Posición de precio ─────────────────────────────
    # Ratios relativas a EMAs: estables en cualquier régimen de precio
    df["close_vs_ema7"]  = df["close"] / (df["ema_7"]  + 1e-9) - 1
    df["close_vs_ema14"] = df["close"] / (df["ema_14"] + 1e-9) - 1
    df["close_vs_ema50"] = df["close"] / (df["ema_50"] + 1e-9) - 1  # [NEW]
    df["return_1"]  = df["close"].pct_change(1)
    df["return_4"]  = df["close"].pct_change(4)   # [NEW]
    df["return_24"] = df["close"].pct_change(24)  # [NEW]
    
    # ── Grupo: Momentum lags ──────────────────────────────────
    for lag in [1, 2, 3, 4]:
        df[f"return_lag_{lag}"] = df["return_1"].shift(lag)
    
    # ── Grupo: Indicadores (ya en calc_indicators) ────────────
    # [NEW] atr_14_norm — ATR normalizado por precio: independiente del nivel de precio
    df["atr_14_norm"] = df["atr_14"] / (df["close"] + 1e-9)
    
    # ── Grupo: Volumen ────────────────────────────────────────
    df["volume_ratio"]    = df["volume"] / (df["volume"].rolling(24).mean() + 1e-9)
    df["volume_ratio_4h"] = df["volume"] / (df["volume"].rolling(4).mean()  + 1e-9)  # [NEW]
    
    # ── Grupo: Estructura de mercado ─────────────────────────
    h24 = df["high"].rolling(24).max()
    l24 = df["low"].rolling(24).min()
    df["range_pos_24h"]   = (df["close"] - l24) / (h24 - l24 + 1e-9)
    df["dist_to_high_24h"]= (h24 - df["close"]) / (df["close"] + 1e-9)  # [NEW]
    df["dist_to_low_24h"] = (df["close"] - l24) / (df["close"] + 1e-9)  # [NEW]
    
    h7d = df["high"].rolling(7 * 24).max()   # [NEW]
    l7d = df["low"].rolling(7 * 24).min()    # [NEW]
    df["range_pos_7d"] = (df["close"] - l7d) / (h7d - l7d + 1e-9)
    
    # Volatilidad de corto y largo plazo
    ret_series          = df["close"].pct_change()
    df["volatility_24h"]= ret_series.rolling(24).std()           # [NEW] vol corto plazo
    vol_7d              = ret_series.rolling(7 * 24).std()
    df["vol_ratio"]     = df["volatility_24h"] / (vol_7d + 1e-9) # [NEW] régimen de volatilidad
    
    # ── Grupo: Tiempo cíclico ─────────────────────────────────
    df["hour_sin"]  = np.sin(2 * np.pi * df["timestamp"].dt.hour / 24)
    df["hour_cos"]  = np.cos(2 * np.pi * df["timestamp"].dt.hour / 24)
    df["dow_sin"]   = np.sin(2 * np.pi * df["timestamp"].dt.dayofweek / 7)
    df["dow_cos"]   = np.cos(2 * np.pi * df["timestamp"].dt.dayofweek / 7)
    df["month_sin"] = np.sin(2 * np.pi * df["timestamp"].dt.month / 12)  # [NEW]
    df["month_cos"] = np.cos(2 * np.pi * df["timestamp"].dt.month / 12)  # [NEW]
    
    return df


def merge_external(df_1h, fg_df):
    """Añade Fear & Greed Index (datos diarios → merge a velas horarias)."""
    df = df_1h.copy()
    df["date"] = df["timestamp"].dt.normalize()
    df = df.merge(fg_df[["date", "fear_greed"]], on="date", how="left")
    df["fear_greed"] = df["fear_greed"].ffill().fillna(50)
    df.drop(columns=["date"], inplace=True)
    return df


def merge_derivatives(df_1h, deriv_df):
    """
    Fusiona las features de derivados (funding rate, OI, L/S ratio)
    con el DataFrame horario 1h usando merge_asof.
    """
    if deriv_df is None or len(deriv_df) == 0:
        for col in ["funding_rate", "funding_rate_24h_avg", "oi_change_4h", "long_short_ratio"]:
            df_1h[col] = 0.0
        return df_1h
    
    deriv_cols = ["timestamp", "funding_rate", "funding_rate_24h_avg",
                  "oi_change_4h", "long_short_ratio"]
    deriv_df = deriv_df[deriv_cols].copy()
    if deriv_df["timestamp"].dt.tz is None:
        deriv_df["timestamp"] = deriv_df["timestamp"].dt.tz_localize("UTC")
    
    df = pd.merge_asof(
        df_1h.sort_values("timestamp"),
        deriv_df.sort_values("timestamp"),
        on="timestamp", direction="backward"
    )
    for col in ["funding_rate", "funding_rate_24h_avg", "oi_change_4h", "long_short_ratio"]:
        df[col] = df[col].ffill().fillna(0.0)
    return df


def merge_multitf(df_1h, df_4h_raw, df_1d_raw):
    """
    Añade features de timeframes 4h y 1d con corrección de lookahead:
    La vela 4h abierta a las 12:00 solo está disponible desde las 16:00.
    La vela diaria del lunes solo está disponible desde el martes 00:00.
    """
    df_4h = calc_indicators(df_4h_raw.copy())
    df_4h["return_4h"]    = df_4h["close"].pct_change(1)
    df_4h_f = df_4h[["timestamp", "rsi_14", "return_4h"]].copy()
    df_4h_f.columns = ["timestamp", "rsi_4h", "return_4h"]
    df_4h_f["timestamp"] = df_4h_f["timestamp"] + pd.Timedelta(hours=4)  # anti-lookahead

    df_1d = calc_indicators(df_1d_raw.copy())
    df_1d["return_1d"] = df_1d["close"].pct_change(1)
    df_1d_f = df_1d[["timestamp", "rsi_14", "return_1d"]].copy()
    df_1d_f.columns = ["timestamp", "rsi_1d", "return_1d"]
    df_1d_f["timestamp"] = df_1d_f["timestamp"] + pd.Timedelta(days=1)  # anti-lookahead

    df = pd.merge_asof(df_1h.sort_values("timestamp"),
                       df_4h_f.sort_values("timestamp"), on="timestamp", direction="backward")
    df = pd.merge_asof(df, df_1d_f.sort_values("timestamp"),
                       on="timestamp", direction="backward")
    return df


def add_btc_cross_features(df, btc_1h_df):
    """
    [NEW] Añade features cross-asset de BTC para ETH/SOL/AVAX.
    BTC domina el mercado crypto: su retorno y RSI tienen poder
    predictivo para las altcoins.
    
    Para BTC mismo, estas features se rellenan con 0 (neutro).
    """
    if btc_1h_df is None:
        df["btc_return_1"]  = 0.0
        df["btc_return_4h"] = 0.0
        df["btc_rsi_14"]    = 0.0
        return df
    
    btc = calc_indicators(btc_1h_df.copy())
    if btc["timestamp"].dt.tz is None:
        btc["timestamp"] = btc["timestamp"].dt.tz_localize("UTC")
    btc["btc_return_1"]  = btc["close"].pct_change(1)
    btc["btc_return_4h"] = btc["close"].pct_change(4)
    btc_f = btc[["timestamp", "btc_return_1", "btc_return_4h", "rsi_14"]].copy()
    btc_f.columns = ["timestamp", "btc_return_1", "btc_return_4h", "btc_rsi_14"]
    
    df = pd.merge_asof(
        df.sort_values("timestamp"),
        btc_f.sort_values("timestamp"),
        on="timestamp", direction="backward"
    )
    df["btc_return_1"]  = df["btc_return_1"].ffill().fillna(0.0)
    df["btc_return_4h"] = df["btc_return_4h"].ffill().fillna(0.0)
    df["btc_rsi_14"]    = df["btc_rsi_14"].ffill().fillna(50.0)
    return df


def add_targets(df, horizons):
    for h in horizons:
        df[f"target_ret_{h}"] = (df["close"].shift(-h) - df["close"]) / df["close"]
    return df


def build_all_datasets(cfg, fg_df, force=False):
    """
    Pipeline completo de construcción de features.
    Para cada moneda:
    1. OHLCV 1h → features técnicas
    2. + Fear & Greed (sentimiento)
    3. + Multi-timeframe (4h, 1d)
    4. + Derivados (funding rate, OI, L/S ratio)
    5. + Cross-asset BTC (solo para altcoins)
    6. + Targets de retorno multi-horizonte
    """
    # Pre-cargar BTC 1h para cross-asset features
    btc_1h_path = f"{cfg['raw_dir']}/BTC_1h_raw.csv"
    if not os.path.exists(btc_1h_path):
        btc_1h_path = f"{cfg['data_dir']}/BTC_1h_raw.csv"
    btc_1h_df = None
    if os.path.exists(btc_1h_path):
        btc_1h_df = pd.read_csv(btc_1h_path, parse_dates=["timestamp"])
        if btc_1h_df["timestamp"].dt.tz is None:
            btc_1h_df["timestamp"] = btc_1h_df["timestamp"].dt.tz_localize("UTC")
    
    for coin in cfg["coins"]:
        out = f"{cfg['data_dir']}/{coin}_hourly.csv"
        if os.path.exists(out) and not force:
            raw_path = f"{cfg['data_dir']}/{coin}_1h_raw.csv"
            if not os.path.exists(raw_path):
                raw_path = f"{cfg['raw_dir']}/{coin}_1h_raw.csv"
            if os.path.exists(raw_path):
                raw_last  = pd.read_csv(raw_path, usecols=["timestamp"],
                                        parse_dates=["timestamp"])["timestamp"].max()
                proc_last = pd.read_csv(out, usecols=["timestamp"],
                                        parse_dates=["timestamp"])["timestamp"].max()
                if raw_last <= proc_last:
                    print(f"  {coin}: features al día ✅"); continue
                print(f"  {coin}: raw más nuevo → regenerando...", end=" ")
            else:
                print(f"  {coin}: features ya existen ✅"); continue
        else:
            print(f"  {coin}: construyendo features...", end=" ")
        if os.path.exists(out): os.remove(out)
        
        # ── OHLCV 1h ─────────────────────────────────────────
        for candidate in [f"{cfg['data_dir']}/{coin}_1h_raw.csv",
                           f"{cfg['raw_dir']}/{coin}_1h_raw.csv"]:
            if os.path.exists(candidate):
                p1h = candidate; break
        df_1h = pd.read_csv(p1h, parse_dates=["timestamp"]).sort_values("timestamp").reset_index(drop=True)
        if df_1h["timestamp"].dt.tz is None:
            df_1h["timestamp"] = df_1h["timestamp"].dt.tz_localize("UTC")
        
        # ── Features técnicas ─────────────────────────────────
        df = add_features_1h(df_1h)
        
        # ── Fear & Greed ──────────────────────────────────────
        df = merge_external(df, fg_df)
        
        # ── Multi-timeframe ───────────────────────────────────
        found_tf = False
        for raw_dir in [cfg["data_dir"], cfg["raw_dir"]]:
            p4h = f"{raw_dir}/{coin}_4h_raw.csv"
            p1d = f"{raw_dir}/{coin}_1d_raw.csv"
            if os.path.exists(p4h) and os.path.exists(p1d):
                df_4h = pd.read_csv(p4h, parse_dates=["timestamp"])
                df_1d = pd.read_csv(p1d, parse_dates=["timestamp"])
                for d in [df_4h, df_1d]:
                    if d["timestamp"].dt.tz is None:
                        d["timestamp"] = d["timestamp"].dt.tz_localize("UTC")
                df = merge_multitf(df, df_4h, df_1d)
                found_tf = True; break
        if not found_tf:
            for col in ["rsi_4h", "return_4h", "rsi_1d", "return_1d"]:
                df[col] = 0.0
        
        # ── Derivados de Binance Futures ──────────────────────
        symbol = COINS_MAP[coin]
        try:
            deriv_df = build_derivatives_df(symbol, coin, cfg["data_dir"])
            df = merge_derivatives(df, deriv_df)
        except Exception as e:
            print(f"\n  ⚠ {coin} derivados no disponibles ({e}) — usando 0")
            for col in ["funding_rate", "funding_rate_24h_avg", "oi_change_4h", "long_short_ratio"]:
                df[col] = 0.0
        
        # ── Cross-asset BTC ───────────────────────────────────
        if coin == "BTC":
            # BTC no necesita cross-asset de sí mismo
            df["btc_return_1"]  = 0.0
            df["btc_return_4h"] = 0.0
            df["btc_rsi_14"]    = 0.0
        else:
            df = add_btc_cross_features(df, btc_1h_df)
        
        # ── Targets multi-horizonte ───────────────────────────
        df = add_targets(df, cfg["horizons"])
        df = df.dropna().reset_index(drop=True)
        df.to_csv(out, index=False)
        last_date = pd.read_csv(out, usecols=["timestamp"])["timestamp"].iloc[-1][:10]
        feat_cols_present = [c for c in cfg["feature_cols"] if c in df.columns]
        print(f"{len(df):,} filas | {len(feat_cols_present)}/{len(cfg['feature_cols'])} features → hasta {last_date} ✅")

print("\nConstruyendo datasets con 44 features...")
build_all_datasets(BASE_CONFIG, FG_DF)
print("✅ Features construidas.")


In [ ]:
def plot_feature_correlation(coin, cfg, n_lags=24):
    path = f"{cfg['data_dir']}/{coin}_{cfg['granularity']}.csv"
    if not os.path.exists(path):
        print(f"  {coin}: dataset no encontrado → ejecutar Feature Engineering primero.")
        return
    df = pd.read_csv(path, parse_dates=["timestamp"]).sort_values("timestamp")
    feat_cols = [c for c in cfg["feature_cols"] if c in df.columns]
    target = "target_ret_1"
    if target not in df.columns:
        print(f"  {coin}: columna {target} no encontrada"); return
    df = df[feat_cols + [target]].dropna().tail(5000)
    corrs = df[feat_cols].corrwith(df[target]).abs().sort_values(ascending=True)

    fig, axes = plt.subplots(1, 2, figsize=(18, 9))
    fig.suptitle(f"{coin} — Correlación Features con Retorno Futuro (1h) — 44 features",
                 fontsize=12, fontweight="bold")
    ax = axes[0]
    colors = ["firebrick" if v > 0.05 else "steelblue" for v in corrs.values]
    ax.barh(corrs.index, corrs.values, color=colors, alpha=0.8)
    ax.axvline(0.05, color="red", linestyle="--", linewidth=1, label="Umbral 5%")
    ax.set_xlabel("|Correlación de Pearson|")
    ax.set_title("Features ordenadas por correlación con target")
    ax.legend(); ax.grid(alpha=0.3, axis="x")

    ax = axes[1]
    top_feats = corrs.tail(15).index.tolist()
    corr_matrix = df[top_feats].corr()
    im = ax.imshow(corr_matrix.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
    ax.set_xticks(range(len(top_feats))); ax.set_yticks(range(len(top_feats)))
    ax.set_xticklabels(top_feats, rotation=45, ha="right", fontsize=7)
    ax.set_yticklabels(top_feats, fontsize=7)
    ax.set_title("Correlación entre Top-15 features\n(rojo=redundancia)")
    plt.colorbar(im, ax=ax, shrink=0.8)
    plt.tight_layout()
    plt.savefig(f"plots/{coin}_feature_corr.png", dpi=120, bbox_inches="tight")
    plt.show()

    high_corr = corrs[corrs > 0.05].sort_values(ascending=False)
    print(f"\n  {coin} — Features con correlación > 5%:")
    for feat, val in high_corr.items():
        print(f"    {feat:30s}: {val:.4f}")

print("\nAnalizando correlaciones feature→target...")
for coin in BASE_CONFIG["coins"]:
    plot_feature_correlation(coin, BASE_CONFIG)
    print()


# 🗃️ Dataset y Preprocesamiento

In [ ]:
class CryptoDataset(Dataset):
    def __init__(self, X: np.ndarray, yr: np.ndarray, noise_std: float = 0.0):
        self.X         = torch.tensor(X,  dtype=torch.float32)
        self.yr        = torch.tensor(yr, dtype=torch.float32)
        self.noise_std = noise_std

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx]
        if self.noise_std > 0:
            x = x + torch.randn_like(x) * self.noise_std
        return x, self.yr[idx]


def load_and_preprocess(coin, cfg):
    path   = f"{cfg['data_dir']}/{coin}_{cfg['granularity']}.csv"
    df     = (pd.read_csv(path, parse_dates=["timestamp"])
                .sort_values("timestamp").reset_index(drop=True))
    feat_cols = [c for c in cfg["feature_cols"] if c in df.columns]
    reg_cols  = [f"target_ret_{h}" for h in cfg["horizons"]]
    aux_cols  = [c for c in ["close"] if c in df.columns and c not in feat_cols]
    df = df[feat_cols + reg_cols + aux_cols].dropna()

    n       = len(df)
    n_train = int(n * cfg["train_ratio"])
    n_val   = int(n * cfg["val_ratio"])
    train_df = df.iloc[:n_train]
    val_df   = df.iloc[n_train : n_train + n_val]
    test_df  = df.iloc[n_train + n_val:]

    feat_scaler = RobustScaler().fit(train_df[feat_cols].values)
    reg_scaler  = RobustScaler().fit(train_df[reg_cols].values)

    def scale_and_window(split_df):
        X  = feat_scaler.transform(split_df[feat_cols].values).astype(np.float32)
        yr = reg_scaler.transform(split_df[reg_cols].values).astype(np.float32)
        seq = cfg["seq_len"]
        Xs, yrs = [], []
        for i in range(seq, len(X)):
            Xs.append(X[i - seq : i]); yrs.append(yr[i])
        return np.array(Xs, dtype=np.float32), np.array(yrs, dtype=np.float32)

    X_tr, yr_tr = scale_and_window(train_df)
    X_va, yr_va = scale_and_window(val_df)
    X_te, yr_te = scale_and_window(test_df)
    close_test  = test_df["close"].values[cfg["seq_len"]:]

    print(f"  {coin} → train: {X_tr.shape}, val: {X_va.shape}, "
          f"test: {X_te.shape} | features: {len(feat_cols)}")

    return (
        {"train": CryptoDataset(X_tr, yr_tr),
         "val":   CryptoDataset(X_va, yr_va),
         "test":  CryptoDataset(X_te, yr_te)},
        feat_scaler, reg_scaler, close_test, feat_cols,
    )


def make_loaders(datasets: dict, cfg: dict) -> dict:
    loaders = {}
    for split, ds in datasets.items():
        is_train = (split == "train")
        noise    = 0.01 if is_train else 0.0
        ds_noise = CryptoDataset(ds.X.numpy(), ds.yr.numpy(), noise_std=noise)
        loaders[split] = DataLoader(
            ds_noise,
            batch_size  = cfg["batch_size"],
            shuffle     = is_train,
            num_workers = 2,
            pin_memory  = (DEVICE.type == "cuda"),
            persistent_workers = True,
        )
    return loaders

print("✅ Dataset definido.")


# 🧠 Modelo LSTM con Atención Temporal

In [ ]:
class CryptoLSTM(nn.Module):
    """
    LSTM apilado con atención temporal aprendida.

    Diagrama del forward pass:
      x: (B, T, F)  — batch, seq_len, n_features (44)
      → input_proj → (B, T, H)       Proyección lineal al espacio oculto
      → LSTM        → (B, T, H)       Todos los estados ocultos
      → attn_pool   → (B, H)         Suma ponderada (atención blanda)
      → norm + drop → (B, H)
      → mlp         → (B, head_hidden//2)
      → head        → (B, n_horizons) Predicciones multi-horizonte (1h,2h,3h,4h)
    """
    def __init__(self, n_features: int, n_horizons: int, cfg: dict):
        super().__init__()
        H      = cfg["lstm_hidden"]
        head_h = cfg["head_hidden"]

        self.input_proj = nn.Sequential(
            nn.Linear(n_features, H),
            nn.LayerNorm(H),
        )
        lstm_drop = cfg["lstm_dropout"] if cfg["lstm_layers"] > 1 else 0.0
        self.lstm = nn.LSTM(
            input_size  = H,
            hidden_size = H,
            num_layers  = cfg["lstm_layers"],
            dropout     = lstm_drop,
            batch_first = True,
        )
        self.attn = nn.Linear(H, 1)
        self.norm = nn.LayerNorm(H)
        self.drop = nn.Dropout(cfg["head_dropout"])
        self.mlp  = nn.Sequential(
            nn.Linear(H, head_h),
            nn.LayerNorm(head_h),
            nn.GELU(),
            nn.Dropout(cfg["head_dropout"]),
            nn.Linear(head_h, head_h // 2),
            nn.GELU(),
        )
        self.head = nn.Linear(head_h // 2, n_horizons)
        self._init_weights()

    def _init_weights(self):
        for name, p in self.lstm.named_parameters():
            if   "weight_ih" in name: nn.init.xavier_uniform_(p.data)
            elif "weight_hh" in name: nn.init.orthogonal_(p.data)
            elif "bias" in name:
                p.data.zero_()
                n = p.size(0)
                p.data[n//4 : n//2].fill_(1.0)   # forget gate bias = 1
        for m in self.modules():
            if isinstance(m, nn.Linear) and m is not self.head:
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.input_proj(x)
        lstm_out, _ = self.lstm(x)
        attn_w  = torch.softmax(self.attn(lstm_out), dim=1)
        context = (attn_w * lstm_out).sum(dim=1)
        context = self.drop(self.norm(context))
        return self.head(self.mlp(context))


class HorizonWeightedMSE(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        hw = torch.tensor(cfg["horizon_weights"], dtype=torch.float32)
        self.register_buffer("hw", hw / hw.sum())

    def forward(self, pred, true):
        return (nn.functional.mse_loss(pred, true, reduction="none") * self.hw).mean()


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print("✅ Modelo LSTM definido.")
print()
print("Estimación de parámetros (BTC, 44 features, seq=24):")
cfg_btc = BEST_CONFIGS["BTC"]
model_test = CryptoLSTM(44, 4, cfg_btc)
print(f"  CryptoLSTM (44 features): {count_params(model_test):>10,} parámetros")


# 🏋️ Entrenamiento

In [ ]:
def train_model(model, loaders, cfg, coin):
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=cfg["learning_rate"], weight_decay=cfg["weight_decay"]
    )
    scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=20, T_mult=2)
    criterion = HorizonWeightedMSE(cfg).to(DEVICE)

    warmup_epochs = 5
    for g in optimizer.param_groups:
        g["lr"] = cfg["learning_rate"] / 10

    best_val, patience_c = float("inf"), 0
    ckpt    = f"{WORK_DIR}/best_{coin}.pt"
    history = {"train_loss": [], "val_loss": []}

    for epoch in range(1, cfg["epochs"] + 1):
        if epoch == warmup_epochs + 1:
            for g in optimizer.param_groups:
                g["lr"] = cfg["learning_rate"]

        model.train()
        t_loss = 0.0
        for X, yr in loaders["train"]:
            X, yr = X.to(DEVICE), yr.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(X), yr)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            t_loss += loss.item()
        t_loss /= len(loaders["train"])

        model.eval()
        v_loss = 0.0
        with torch.no_grad():
            for X, yr in loaders["val"]:
                X, yr = X.to(DEVICE), yr.to(DEVICE)
                v_loss += criterion(model(X), yr).item()
        v_loss /= len(loaders["val"])

        if epoch > warmup_epochs:
            scheduler.step(epoch - warmup_epochs)

        history["train_loss"].append(t_loss)
        history["val_loss"].append(v_loss)

        if epoch % 10 == 0 or epoch == 1:
            print(f"  Epoch {epoch:3d}/{cfg['epochs']} | "
                  f"Train {t_loss:.5f} | Val {v_loss:.5f}")

        if v_loss < best_val:
            best_val = v_loss; patience_c = 0
            torch.save(model.state_dict(), ckpt)
        else:
            patience_c += 1
            if patience_c >= cfg["patience"]:
                print(f"  ⏹ Early stopping epoch {epoch} (best val: {best_val:.5f})")
                break

    model.load_state_dict(torch.load(ckpt, map_location=DEVICE, weights_only=True))
    return history

print("✅ Entrenamiento definido.")


# 📊 Evaluación y Métricas Honestas

In [ ]:
def evaluate_model(model, loader, close_test, reg_scaler, horizons, coin):
    model.eval()
    rp_all, rt_all = [], []
    with torch.no_grad():
        for X, yr in loader:
            rp_all.append(model(X.to(DEVICE)).cpu().numpy())
            rt_all.append(yr.numpy())

    reg_pred = reg_scaler.inverse_transform(np.concatenate(rp_all))
    reg_true = reg_scaler.inverse_transform(np.concatenate(rt_all))

    def to_price(arr, idx):
        n = min(len(close_test), len(arr))
        return close_test[:n] * (1 + arr[:n, idx])

    results = []
    print(f"\n{'─'*65}")
    print(f"  {coin} — Resultados Test")
    print(f"{'─'*65}")
    print(f"  {'H':>4} | {'MAE USD':>10} | {'RMSE USD':>10} | {'MAPE%':>7} | {'DirAcc':>7}")
    print(f"  {'─'*4}-+-{'─'*10}-+-{'─'*10}-+-{'─'*7}-+-{'─'*7}")

    for i, h in enumerate(horizons):
        pp   = to_price(reg_pred, i)
        pt   = to_price(reg_true, i)
        n    = len(pp)
        mae  = mean_absolute_error(pt, pp)
        rmse = math.sqrt(mean_squared_error(pt, pp))
        mape = float(np.mean(np.abs((pt-pp)/(np.abs(pt)+1e-9))))*100
        dacc = float(np.mean(np.sign(reg_pred[:n,i]) == np.sign(reg_true[:n,i])))
        print(f"  {h:>2}h  | {mae:>10.2f} | {rmse:>10.2f} | {mape:>6.2f}% | {dacc:>7.3f}")
        results.append({"coin":coin,"horizon_h":h,"MAE":mae,"RMSE":rmse,"MAPE":mape,"DirAcc":dacc})

    return pd.DataFrame(results), reg_pred, reg_true


def moving_average_baseline(df_test, horizons, window=4):
    close, results = df_test["close"].values, []
    for h in horizons:
        p = [close[i-window:i].mean() for i in range(window, len(close)-h)]
        t = [close[i+h]               for i in range(window, len(close)-h)]
        p, t = np.array(p), np.array(t)
        results.append({"model":"Baseline","horizon_h":h,
                        "MAE": mean_absolute_error(t,p),
                        "RMSE":math.sqrt(mean_squared_error(t,p)),
                        "MAPE":float(np.mean(np.abs((t-p)/(np.abs(t)+1e-9))))*100})
    return pd.DataFrame(results)


def plot_attention_weights(model, loader, coin, n_samples=3):
    model.eval()
    fig, axes = plt.subplots(1, n_samples, figsize=(14, 3))
    fig.suptitle(f"{coin} — Pesos de Atención Temporal", fontsize=11, fontweight="bold")
    sample_idx = 0
    for X, _ in loader:
        if sample_idx >= n_samples: break
        x = X[0:1].to(DEVICE)
        with torch.no_grad():
            proj_x   = model.input_proj(x)
            lstm_out, _ = model.lstm(proj_x)
            attn_w   = torch.softmax(model.attn(lstm_out), dim=1).squeeze().cpu().numpy()
        ax = axes[sample_idx] if n_samples > 1 else axes
        ax.bar(range(len(attn_w)), attn_w, color="steelblue", alpha=0.7, width=1.0)
        ax.set_xlabel("Paso temporal"); ax.set_ylabel("Peso"); ax.set_title(f"Muestra {sample_idx+1}")
        ax.grid(alpha=0.3); sample_idx += 1
    plt.tight_layout()
    plt.savefig(f"plots/{coin}_attention.png", dpi=120, bbox_inches="tight")
    plt.show()


def compute_diracc_full(reg_pred, reg_true):
    results = {}
    for i in range(reg_pred.shape[1]):
        dacc = float(np.mean(np.sign(reg_pred[:, i]) == np.sign(reg_true[:, i])))
        results[f"h{i+1}"] = dacc
        print(f"  DirAcc H{i+1} (sin filtro): {dacc*100:.2f}%  "
              f"{'✅ por encima del azar' if dacc > 0.52 else '⚠ cerca del azar'}")
    return results


def compute_profit_factor(trades: np.ndarray) -> float:
    wins   = trades[trades > 0].sum()
    losses = np.abs(trades[trades < 0].sum())
    return float(wins / (losses + 1e-9))


def print_honest_metrics(coin, dacc_dict, bt_result):
    print(f"\n{'='*60}")
    print(f"  MÉTRICAS HONESTAS — {coin}")
    print(f"{'='*60}")
    print(f"\n  [PRIMARIA] DirAcc sin filtro:")
    for h, val in dacc_dict.items():
        bar = "█" * int(val * 50)
        print(f"    {h}: {val*100:.2f}%  {bar}")
    if bt_result is not None:
        trades = bt_result["retornos"]
        pf     = compute_profit_factor(trades)
        print(f"\n  [SECUNDARIA] Backtesting:")
        print(f"    Trades:        {len(trades)}")
        print(f"    Win rate:      {bt_result['win_rate']*100:.1f}%")
        print(f"    Profit factor: {pf:.2f}  {'✅' if pf > 1.5 else '⚠'}")
        print(f"    Retorno total: {bt_result['ret_total']*100:.2f}%")
        print(f"    Sharpe:        {bt_result['sharpe']:.2f}")

print("✅ Evaluación definida.")


# 📈 Visualizaciones

In [ ]:
def plot_history(history, coin):
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(history["train_loss"], label="Train", linewidth=1.5)
    ax.plot(history["val_loss"],   label="Val",   linewidth=1.5)
    ax.set_title(f"{coin} — Curva de Pérdida (LSTM, 44 features)")
    ax.legend(); ax.grid(alpha=0.3)
    ax.set_xlabel("Época"); ax.set_ylabel("HorizonWeightedMSE")
    plt.tight_layout()
    plt.savefig(f"plots/{coin}_loss.png", dpi=120, bbox_inches="tight")
    plt.show()

def plot_preds(reg_pred, reg_true, close_test, coin, horizons, n=200):
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    for idx, (ax, h) in enumerate(zip(axes.flatten(), horizons)):
        nn_ = min(n, len(close_test))
        ax.plot(close_test[:nn_]*(1+reg_true[:nn_,idx]), label="Real",     alpha=0.85, linewidth=1.2)
        ax.plot(close_test[:nn_]*(1+reg_pred[:nn_,idx]), label="Predicho", alpha=0.85, linewidth=1.2, linestyle="--")
        ax.set_title(f"{coin} — {h}h"); ax.legend(); ax.grid(alpha=0.3)
    plt.suptitle(f"{coin} — Precio real vs predicho (44 features)", fontsize=12)
    plt.tight_layout()
    plt.savefig(f"plots/{coin}_pred.png", dpi=120, bbox_inches="tight")
    plt.show()

def plot_comparison(lstm_res, base_res, coin):
    lr = lstm_res[["horizon_h","MAE","RMSE","MAPE"]].copy(); lr["model"] = "LSTM 44-feat"
    combined = pd.concat([lr, base_res[["model","horizon_h","MAE","RMSE","MAPE"]]])
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for ax, m in zip(axes, ["MAE","RMSE","MAPE"]):
        for name, grp in combined.groupby("model"):
            ax.plot(grp["horizon_h"], grp[m], marker="o", label=name, linewidth=2)
        ax.set_title(f"{m} — {coin}"); ax.legend(); ax.grid(alpha=0.3)
        ax.set_xlabel("Horizonte (horas)")
    plt.tight_layout()
    plt.savefig(f"plots/{coin}_cmp.png", dpi=120, bbox_inches="tight")
    plt.show()

print("✅ Visualizaciones definidas.")


# 🔄 Walk-Forward Validation

In [ ]:
def walk_forward_validation(coin: str, cfg: dict, n_folds: int = 5, comision: float = 0.001):
    print(f"\n{'='*60}")
    print(f"  WALK-FORWARD VALIDATION — {coin} ({n_folds} folds)")
    print(f"{'='*60}")

    path = f"{cfg['data_dir']}/{coin}_{cfg['granularity']}.csv"
    if not os.path.exists(path):
        print(f"  ⚠ {path} no encontrado"); return pd.DataFrame()

    df        = pd.read_csv(path, parse_dates=["timestamp"]).sort_values("timestamp").reset_index(drop=True)
    feat_cols = [c for c in cfg["feature_cols"] if c in df.columns]
    reg_cols  = [f"target_ret_{h}" for h in cfg["horizons"]]
    df        = df[feat_cols + reg_cols + ["close"]].dropna().reset_index(drop=True)
    n         = len(df)

    holdout_start = int(n * 0.85)
    df_wf         = df.iloc[:holdout_start].reset_index(drop=True)
    n_wf          = len(df_wf)

    fold_size = n_wf // (n_folds + 1)
    val_size  = max(fold_size // 3, cfg["seq_len"] * 5)
    test_size = max(fold_size // 3, cfg["seq_len"] * 5)
    min_train = cfg["seq_len"] * 10

    fold_results = []

    for fold in range(n_folds):
        test_end   = n_wf - fold * fold_size
        test_start = max(test_end - test_size, min_train + val_size)
        val_start  = max(test_start - val_size, min_train)
        train_end  = val_start

        if train_end < min_train:
            print(f"  Fold {fold+1}: datos insuficientes, omitiendo"); continue

        train_df = df_wf.iloc[:train_end].reset_index(drop=True)
        val_df   = df_wf.iloc[val_start:test_start].reset_index(drop=True)
        test_df  = df_wf.iloc[test_start:test_end].reset_index(drop=True)

        print(f"\n  Fold {fold+1}/{n_folds} | train: {len(train_df)} | val: {len(val_df)} | test: {len(test_df)}")
        if len(val_df) < cfg["seq_len"] * 2 or len(test_df) < cfg["seq_len"] * 2:
            print(f"  Fold {fold+1}: segmentos demasiado pequeños, omitiendo"); continue

        feat_scaler = RobustScaler().fit(train_df[feat_cols].values)
        reg_scaler  = RobustScaler().fit(train_df[reg_cols].values)

        def make_windows(split_df):
            X_sc  = feat_scaler.transform(split_df[feat_cols].values).astype(np.float32)
            yr_sc = reg_scaler.transform(split_df[reg_cols].values).astype(np.float32)
            seq   = cfg["seq_len"]
            Xs, yrs = [], []
            for i in range(seq, len(X_sc)):
                Xs.append(X_sc[i-seq:i]); yrs.append(yr_sc[i])
            if not Xs:
                return np.empty((0, cfg["seq_len"], len(feat_cols))), np.empty((0, len(reg_cols)))
            return np.array(Xs, dtype=np.float32), np.array(yrs, dtype=np.float32)

        X_tr, yr_tr = make_windows(train_df)
        X_va, yr_va = make_windows(val_df)
        X_te, yr_te = make_windows(test_df)
        if len(X_tr) == 0 or len(X_te) == 0: continue

        n_feat = X_tr.shape[2]
        model  = CryptoLSTM(n_feat, len(cfg["horizons"]), cfg).to(DEVICE)
        datasets = {
            "train": CryptoDataset(X_tr, yr_tr, noise_std=0.01),
            "val":   CryptoDataset(X_va, yr_va),
            "test":  CryptoDataset(X_te, yr_te),
        }
        loaders = {
            split: DataLoader(ds, batch_size=cfg["batch_size"],
                              shuffle=(split == "train"), num_workers=2,
                              pin_memory=(DEVICE.type == "cuda"))
            for split, ds in datasets.items()
        }
        cfg_fold = {**cfg, "epochs": 60, "patience": 10}
        history  = train_model(model, loaders, cfg_fold, f"{coin}_fold{fold+1}")

        model.eval()
        rp_all, rt_all = [], []
        with torch.no_grad():
            for X, yr in loaders["test"]:
                rp_all.append(model(X.to(DEVICE)).cpu().numpy())
                rt_all.append(yr.numpy())
        reg_pred = reg_scaler.inverse_transform(np.concatenate(rp_all))
        reg_true = reg_scaler.inverse_transform(np.concatenate(rt_all))

        row = {"fold": fold + 1, "n_test": len(reg_pred)}
        for i, h in enumerate(cfg["horizons"]):
            dacc = float(np.mean(np.sign(reg_pred[:, i]) == np.sign(reg_true[:, i])))
            row[f"DirAcc_H{h}"] = dacc

        train_prices = train_df["close"].values.ravel()
        tr_returns   = np.diff(train_prices) / (train_prices[:-1] + 1e-9)
        sigma_4h     = float(np.std(tr_returns)) * np.sqrt(4)
        umbral       = float(np.clip(sigma_4h * 0.8, 0.001, 0.006))
        row["umbral_train"] = umbral

        close_test_fold = test_df["close"].values[cfg["seq_len"]:]
        n_bt = min(len(close_test_fold) - 4, len(reg_pred) - 4)
        trades_fold = []; cd = 0
        for i in range(n_bt - 4):
            if cd > 0: cd -= 1; continue
            pb = (close_test_fold[min(i+4, n_bt-1)] - close_test_fold[i]) / close_test_fold[i]
            if reg_pred[i, 3] > umbral:
                trades_fold.append(pb - 2 * comision); cd = 2
            elif reg_pred[i, 3] < -umbral:
                trades_fold.append(-pb - 2 * comision); cd = 2

        if trades_fold:
            ta = np.array(trades_fold)
            row["win_rate"]      = float(np.mean(ta > 0))
            row["n_trades"]      = len(ta)
            row["ret_total"]     = float(np.prod(1 + ta) - 1)
            row["profit_factor"] = compute_profit_factor(ta)
        else:
            row["win_rate"] = None; row["n_trades"] = 0
            row["ret_total"] = 0.0; row["profit_factor"] = None

        fold_results.append(row)
        print(f"  Fold {fold+1} → "
              + " | ".join(f"DirAcc_H{h}: {row[f'DirAcc_H{h}']*100:.1f}%"
                            for h in cfg["horizons"])
              + (f" | WinRate: {row['win_rate']*100:.1f}%" if row['win_rate'] else ""))

    if not fold_results: return pd.DataFrame()
    df_res = pd.DataFrame(fold_results)

    print(f"\n  {'─'*60}")
    print(f"  RESUMEN WALK-FORWARD — {coin}")
    print(f"  {'─'*60}")
    for h in cfg["horizons"]:
        col  = f"DirAcc_H{h}"
        mean = df_res[col].mean(); std = df_res[col].std()
        print(f"  DirAcc H{h}: {mean*100:.2f}% ± {std*100:.2f}%  "
              f"{'✅ habilidad real' if mean > 0.52 else '⚠ cerca del azar'}")
    return df_res


def run_all_wf(coins, configs, n_folds=5):
    all_wf = {}
    for coin in coins:
        cfg = configs.get(coin, BASE_CONFIG)
        try:
            all_wf[coin] = walk_forward_validation(coin, cfg, n_folds=n_folds)
        except Exception:
            import traceback; traceback.print_exc()
    return all_wf


# 🚀 Pipeline Principal + Backtesting

In [ ]:
COMISION = 0.001  # 0.1% por operación (estándar Binance)

def calibrate_threshold_on_train(coin: str, cfg: dict) -> float:
    path = f"{cfg['data_dir']}/{coin}_{cfg['granularity']}.csv"
    df   = pd.read_csv(path, parse_dates=["timestamp"]).sort_values("timestamp")
    n_tr = int(len(df) * cfg["train_ratio"])
    train_prices = df["close"].values[:n_tr]
    returns      = np.diff(train_prices) / train_prices[:-1]
    sigma_4h     = float(np.std(returns)) * np.sqrt(4)
    umbral       = float(np.clip(sigma_4h * 0.8, 0.001, 0.006))
    print(f"  {coin}: umbral calibrado en TRAIN = {umbral*100:.3f}%  "
          f"(σ_4h_train = {sigma_4h*100:.3f}%)")
    return umbral


def save_artifacts(coin, model, feat_scaler, reg_scaler, feat_cols):
    torch.save(model.state_dict(), f"{WORK_DIR}/model_{coin}.pt")
    with open(f"{WORK_DIR}/scaler_{coin}.pkl", "wb") as f:
        pickle.dump({"feat_scaler": feat_scaler, "reg_scaler": reg_scaler,
                     "feat_cols": feat_cols, "n_features": len(feat_cols)}, f)
    print(f"  💾 model_{coin}.pt + scaler_{coin}.pkl guardados")


def load_artifacts(coin, cfg):
    model_path  = f"{MODELS_DIR}/model_{coin}.pt"
    scaler_path = f"{MODELS_DIR}/scaler_{coin}.pkl"
    if not os.path.exists(model_path) or not os.path.exists(scaler_path): return None
    try:
        with open(scaler_path, "rb") as f:
            artifacts = pickle.load(f)
        feat_cols = artifacts["feat_cols"]
        n_saved   = artifacts["n_features"]
        csv_path  = f"{cfg['data_dir']}/{coin}_{cfg['granularity']}.csv"
        df_cols   = pd.read_csv(csv_path, nrows=1).columns.tolist()
        feat_available = [c for c in feat_cols if c in df_cols]
        if len(feat_available) != n_saved:
            print(f"  ⚠ {coin}: features incompatibles ({n_saved} guardadas vs {len(feat_available)} disponibles) → reentrenando")
            return None
        model = CryptoLSTM(n_saved, len(cfg["horizons"]), cfg).to(DEVICE)
        model.load_state_dict(torch.load(model_path, map_location=DEVICE, weights_only=True))
        model.eval()
        print(f"  ✅ {coin}: modelo cargado ({n_saved} features)")
        return model, artifacts["feat_scaler"], artifacts["reg_scaler"], feat_cols
    except Exception as e:
        print(f"  ⚠ {coin}: error cargando ({e}) → reentrenando")
        return None


def run_backtest_fixed(coin, result, cfg, cooldown_velas=2):
    umbral_ret = calibrate_threshold_on_train(coin, cfg)
    reg_pred   = result["reg_pred"]
    close_test = result["close_test"]
    h_idx      = 3  # horizonte 4h
    n          = min(len(close_test) - 4, len(reg_pred) - 4)
    precios    = close_test[:n]
    señales    = reg_pred[:n, h_idx]
    capital    = 1000.0
    trades     = []; cooldown = 0; n_long = n_short = 0

    for i in range(n - 4):
        if cooldown > 0: cooldown -= 1; continue
        p_entrada = precios[i]
        p_salida  = precios[min(i + 4, n - 1)]
        ret_bruto = (p_salida - p_entrada) / p_entrada
        if señales[i] > umbral_ret:
            ret_neto = ret_bruto - 2 * COMISION
            capital *= (1 + ret_neto); trades.append(ret_neto)
            cooldown = cooldown_velas; n_long += 1
        elif señales[i] < -umbral_ret:
            ret_neto = -ret_bruto - 2 * COMISION
            capital *= (1 + ret_neto); trades.append(ret_neto)
            cooldown = cooldown_velas; n_short += 1

    ret_hold = (precios[-1] - precios[0]) / precios[0]
    if len(trades) == 0:
        print(f"  {coin}: sin trades con umbral {umbral_ret*100:.3f}%"); return None

    trades_arr = np.array(trades)
    win_rate   = float(np.mean(trades_arr > 0))
    ret_medio  = float(np.mean(trades_arr))
    ret_total  = float(capital / 1000.0 - 1)
    sharpe     = (ret_medio / (trades_arr.std() + 1e-9)) * np.sqrt(252 * 6)
    pf         = compute_profit_factor(trades_arr)

    print(f"\n  {coin} — Backtesting (umbral TRAIN {umbral_ret*100:.3f}%, cooldown {cooldown_velas}v)")
    print(f"  {'─'*60}")
    print(f"    Trades:              {len(trades):4d}  (LONG: {n_long}, SHORT: {n_short})")
    print(f"    Win rate:            {win_rate*100:.1f}%")
    print(f"    Profit factor:       {pf:.2f}")
    print(f"    Retorno medio/trade: {ret_medio*100:.3f}% (neto)")
    print(f"    Retorno total:       {ret_total*100:.2f}% (neto)")
    print(f"    Sharpe (anual.):     {sharpe:.2f}")
    print(f"    Hold (buy & hold):   {ret_hold*100:.2f}%")
    if len(trades) < 30: print(f"    ⚠ Solo {len(trades)} trades — baja significancia estadística")
    print(f"    {'✅ Modelo supera hold' if ret_total > ret_hold else '⚠  Hold supera al modelo'}")

    return {"coin": coin, "n_trades": len(trades), "win_rate": win_rate,
            "ret_total": ret_total, "ret_hold": ret_hold, "sharpe": sharpe,
            "profit_factor": pf, "retornos": trades_arr}


def run_pipeline(coin, cfg):
    print(f"\n{'='*60}")
    print(f"  {coin.upper()} | seq={cfg['seq_len']} hidden={cfg['lstm_hidden']} "
          f"layers={cfg['lstm_layers']} lr={cfg['learning_rate']:.2e}")
    print(f"{'='*60}")

    loaded = load_artifacts(coin, cfg)

    if loaded is not None:
        model, feat_scaler, reg_scaler, feat_cols = loaded
        path     = f"{cfg['data_dir']}/{coin}_{cfg['granularity']}.csv"
        df       = pd.read_csv(path, parse_dates=["timestamp"]).sort_values("timestamp").reset_index(drop=True)
        reg_cols = [f"target_ret_{h}" for h in cfg["horizons"]]
        aux_cols = [c for c in ["close"] if c in df.columns and c not in feat_cols]
        df       = df[feat_cols + reg_cols + aux_cols].dropna()
        n        = len(df)
        test_df  = df.iloc[int(n*cfg["train_ratio"]) + int(n*cfg["val_ratio"]):]
        seq      = cfg["seq_len"]
        X_te  = feat_scaler.transform(test_df[feat_cols].values).astype(np.float32)
        yr_te = reg_scaler.transform(test_df[reg_cols].values).astype(np.float32)
        Xs, yrs = [], []
        for i in range(seq, len(X_te)):
            Xs.append(X_te[i-seq:i]); yrs.append(yr_te[i])
        X_te = np.array(Xs, dtype=np.float32); yr_te = np.array(yrs, dtype=np.float32)
        close_test  = test_df["close"].values[seq:]
        test_ds     = CryptoDataset(X_te, yr_te)
        test_loader = DataLoader(test_ds, batch_size=cfg["batch_size"], shuffle=False,
                                 num_workers=2, pin_memory=(DEVICE.type=="cuda"),
                                 persistent_workers=True)
        history = None
    else:
        datasets, feat_scaler, reg_scaler, close_test, feat_cols = load_and_preprocess(coin, cfg)
        n_features  = datasets["train"].X.shape[2]
        loaders     = make_loaders(datasets, cfg)
        test_loader = loaders["test"]
        model       = CryptoLSTM(n_features, len(cfg["horizons"]), cfg).to(DEVICE)
        print(f"  Parámetros: {count_params(model):,} | Features: {n_features}")
        history = train_model(model, loaders, cfg, coin)
        plot_history(history, coin)
        save_artifacts(coin, model, feat_scaler, reg_scaler, feat_cols)

    lstm_res, reg_pred, reg_true = evaluate_model(
        model, test_loader, close_test, reg_scaler, cfg["horizons"], coin
    )
    if history is not None:
        plot_preds(reg_pred, reg_true, close_test, coin, cfg["horizons"])
    plot_attention_weights(model, test_loader, coin)

    path_proc = f"{cfg['data_dir']}/{coin}_{cfg['granularity']}.csv"
    df_proc   = pd.read_csv(path_proc, parse_dates=["timestamp"]).sort_values("timestamp")
    n_proc    = len(df_proc)
    n_skip_p  = int(n_proc * cfg["train_ratio"]) + int(n_proc * cfg["val_ratio"])
    base_res  = moving_average_baseline(df_proc.iloc[n_skip_p:].copy(), cfg["horizons"])
    plot_comparison(lstm_res, base_res, coin)

    return {"model": model, "feat_scaler": feat_scaler, "reg_scaler": reg_scaler,
            "lstm_res": lstm_res, "base_res": base_res, "close_test": close_test,
            "cfg": cfg, "feat_cols": feat_cols, "reg_pred": reg_pred, "reg_true": reg_true}


# ── EJECUCIÓN PRINCIPAL ───────────────────────────────────────────────────────
all_results = {}
for coin in BASE_CONFIG["coins"]:
    cfg = BEST_CONFIGS.get(coin, BASE_CONFIG)
    try:
        all_results[coin] = run_pipeline(coin, cfg)
    except FileNotFoundError as e:
        print(f"  ⚠ {coin}: {e}")
    except Exception:
        import traceback; traceback.print_exc()

if all_results:
    print("\n\n" + "="*60)
    print("  WALK-FORWARD VALIDATION (métricas honestas)")
    print("="*60)
    wf_results = run_all_wf(BASE_CONFIG["coins"], BEST_CONFIGS, n_folds=5)

bt_results = {}
for coin, res in all_results.items():
    bt_results[coin] = run_backtest_fixed(coin, res, BEST_CONFIGS[coin])

for coin, res in all_results.items():
    dacc = compute_diracc_full(res["reg_pred"], res["reg_true"])
    print_honest_metrics(coin, dacc, bt_results.get(coin))


# 🔮 Inferencia en Tiempo Real con Intervalos de Confianza

In [ ]:
def get_recent_data(coin, cfg):
    """
    Descarga datos recientes para inferencia, incluyendo todas las
    features del nuevo conjunto de 44 variables.
    """
    seq_len = cfg["seq_len"]; buffer = 60; symbol = COINS_MAP[coin]
    
    # OHLCV 1h
    df_1h = download_recent(symbol, "1h", seq_len + buffer)
    if df_1h is None:
        COIN_IDS = {"BTC":"bitcoin","ETH":"ethereum","SOL":"solana","AVAX":"avalanche-2"}
        days = math.ceil((seq_len + buffer) / 24) + 2
        resp = requests.get(
            f"https://api.coingecko.com/api/v3/coins/{COIN_IDS[coin]}/ohlc",
            params={"vs_currency":"usd","days":days}, timeout=20
        )
        rows = [{"timestamp": pd.to_datetime(c[0], unit="ms", utc=True),
                 "open":float(c[1]),"high":float(c[2]),"low":float(c[3]),
                 "close":float(c[4]),"volume":0.0} for c in resp.json()]
        df_1h = pd.DataFrame(rows).sort_values("timestamp").reset_index(drop=True)
    df_1h["coin"] = coin
    
    # Features técnicas (incluye todas las nuevas)
    df = add_features_1h(df_1h)
    
    # Fear & Greed
    fg_rec = download_fear_greed(limit=10)
    df = merge_external(df, fg_rec)
    
    # Multi-timeframe
    df_4h = download_recent(symbol, "4h", 100)
    df_1d = download_recent(symbol, "1d", 30)
    if df_4h is not None and df_1d is not None:
        df = merge_multitf(df, df_4h, df_1d)
    else:
        for col in ["rsi_4h", "return_4h", "rsi_1d", "return_1d"]:
            df[col] = 0.0
    
    # Derivados recientes
    try:
        deriv_recent = build_derivatives_df(symbol, coin, WORK_DIR, force=True)
        df = merge_derivatives(df, deriv_recent)
    except Exception:
        for col in ["funding_rate", "funding_rate_24h_avg", "oi_change_4h", "long_short_ratio"]:
            df[col] = 0.0
    
    # Cross-asset BTC (para altcoins)
    if coin != "BTC":
        df_btc = download_recent("BTCUSDT", "1h", seq_len + buffer)
        df = add_btc_cross_features(df, df_btc)
    else:
        df["btc_return_1"] = 0.0; df["btc_return_4h"] = 0.0; df["btc_rsi_14"] = 0.0
    
    return df.dropna().reset_index(drop=True)


def predict_next(coin, result):
    model       = result["model"]
    feat_scaler = result["feat_scaler"]
    reg_scaler  = result["reg_scaler"]
    cfg         = result["cfg"]
    feat_cols   = result["feat_cols"]
    seq_len     = cfg["seq_len"]
    mape        = MAPE_HISTORICO.get(coin, 0.01)

    df_recent  = get_recent_data(coin, cfg)
    if len(df_recent) < seq_len:
        raise ValueError(f"Datos insuficientes: {len(df_recent)} < {seq_len}")

    available  = [c for c in feat_cols if c in df_recent.columns]
    missing    = [c for c in feat_cols if c not in df_recent.columns]
    if missing: print(f"  ⚠ {coin}: features ausentes en inferencia: {missing}")
    
    X_raw      = df_recent[available].iloc[-seq_len:].values.astype(np.float32)
    X_tensor   = torch.tensor(feat_scaler.transform(X_raw)[np.newaxis], dtype=torch.float32).to(DEVICE)
    last_close = float(df_recent["close"].iloc[-1])
    last_time  = df_recent["timestamp"].iloc[-1]

    model.eval()
    with torch.no_grad():
        reg_vals = reg_scaler.inverse_transform(model(X_tensor).cpu().numpy())[0]

    preds = {}
    for i, h in enumerate(cfg["horizons"]):
        ret    = float(reg_vals[i])
        price  = round(last_close * (1 + ret), 4)
        margen = price * mape
        preds[f"{h}h"] = {
            "precio_estimado":    price,
            "precio_min":         round(price - margen, 4),
            "precio_max":         round(price + margen, 4),
            "retorno_esperado_%": round(ret * 100, 3),
            "direccion":          "↑ SUBE" if ret > 0 else "↓ BAJA",
        }
    return last_close, last_time, preds


print("=" * 62)
print("  PREDICCIONES EN TIEMPO REAL — LSTM 44 Features")
print("=" * 62)
all_predictions = {}
for coin, res in all_results.items():
    try:
        last_close, last_time, preds = predict_next(coin, res)
        all_predictions[coin] = preds
        print(f"\n  {coin} | Cierre: {last_close:.4f} USD | {last_time}")
        print(f"  {'─'*60}")
        print(f"  {'Horizonte':>10} | {'Estimado':>12} | {'Min':>12} | {'Max':>12} | Dir")
        print(f"  {'─'*10}-+-{'─'*12}-+-{'─'*12}-+-{'─'*12}-+----")
        for h, p in preds.items():
            arrow = "↑" if p["direccion"] == "↑ SUBE" else "↓"
            print(f"  {h:>10} | {p['precio_estimado']:>12.4f} | "
                  f"{p['precio_min']:>12.4f} | {p['precio_max']:>12.4f} | {arrow} "
                  f"{p['retorno_esperado_%']:+.3f}%")
    except Exception as e:
        print(f"  ✗ {coin}: {e}")
        import traceback; traceback.print_exc()


# 📋 Resumen Final Honesto

In [ ]:
def resumen_final_honesto(all_results, bt_results, wf_results):
    print("\n" + "="*65)
    print("  RESUMEN FINAL HONESTO — 44 Features | Orden por fiabilidad")
    print("="*65)
    print("""
  JERARQUÍA DE MÉTRICAS (de más a menos fiable):
  ─────────────────────────────────────────────────────────────
  1. DirAcc walk-forward  → más honesta, varianza entre folds
  2. DirAcc test fijo     → sin filtros, señal real del modelo
  3. Win rate backtest    → con umbral calibrado en TRAIN
  4. Retorno total        → depende del período (sesgo de mercado)
  ─────────────────────────────────────────────────────────────

  GRUPOS DE FEATURES INCLUIDOS EN ESTE MODELO:
  ─────────────────────────────────────────────────────────────
  • Posición de precio (6): close_vs_ema7/14/50, return_1/4/24
  • Momentum lags (4):      return_lag_1/2/3/4
  • Indicadores (8):        rsi_14/6, macd+hist, bb, atr_norm
  • Volumen (2):            volume_ratio, volume_ratio_4h
  • Estructura (6):         range_pos_24h/7d, dist_high/low, vol_ratio
  • Tiempo (6):             hour/dow/month sin+cos
  • Multi-TF (4):           return/rsi 4h y 1d
  • Derivados (4):          funding_rate, funding_24h_avg, oi_change, l/s
  • Sentimiento (1):        fear_greed
  • Cross-asset (3):        btc_return_1/4h, btc_rsi_14
  Total: 44 features curadas
  ─────────────────────────────────────────────────────────────
    """)

    for coin in all_results.keys():
        res = all_results[coin]
        print(f"\n  {coin}")
        print(f"  {'─'*55}")

        if coin in wf_results and not wf_results[coin].empty:
            df_wf = wf_results[coin]
            for h in BASE_CONFIG["horizons"]:
                col = f"DirAcc_H{h}"
                if col in df_wf.columns:
                    m, s = df_wf[col].mean(), df_wf[col].std()
                    flag = "✅" if m > 0.52 else "⚠"
                    print(f"  [WF]   DirAcc H{h}: {m*100:.1f}% ± {s*100:.1f}%  {flag}")

        reg_pred = res["reg_pred"]; reg_true = res["reg_true"]
        for i, h in enumerate(BASE_CONFIG["horizons"]):
            dacc = float(np.mean(np.sign(reg_pred[:, i]) == np.sign(reg_true[:, i])))
            flag = "✅" if dacc > 0.52 else "⚠"
            print(f"  [TEST] DirAcc H{h}: {dacc*100:.1f}%  {flag}")

        bt = bt_results.get(coin)
        if bt:
            pf = bt.get("profit_factor", compute_profit_factor(bt["retornos"]))
            print(f"  [BT]   Win rate: {bt['win_rate']*100:.1f}%  PF: {pf:.2f}  Trades: {bt['n_trades']}")
            if bt["n_trades"] < 30:
                print(f"  ⚠ Pocos trades — no estadísticamente significativo")

    print("\n" + "="*65)
    print("  INTERPRETACIÓN")
    print("="*65)
    print("""
      DirAcc > 54% en walk-forward  → habilidad real demostrada
      DirAcc 52–54%                 → marginal, más validación
      DirAcc < 52%                  → sin señal útil
      Profit Factor > 1.5           → sistema con ventaja neta
    """)

resumen_final_honesto(all_results, bt_results, wf_results if 'wf_results' in dir() else {})
